# Assignment 02 - Colorectal Histopathology CNN Classifier

CAME | PyTorch + Google Colab T4 GPU | custom CNN trained **from scratch** on `1aurent/NCT-CRC-HE` (`NCT_CRC_HE_100K`).

**How this notebook works**

1. Sections 2-9 write `assignment02.py` one part at a time with `%%writefile`. The submitted `.py` is therefore exactly the code in this notebook.
2. Section 10 imports the module and runs the CMS gate checks locally (>= 3 `Conv2d`, allowed layers only, logits `(N, 9)`, 25K-500K parameters).
3. Sections 11-12 train on the GPU with MLflow tracking. The script then verifies `load_trained_model(cpu)` and writes `assignment02_submission.py` with `TRAINED_MODEL_B64` filled in.
4. Sections 13-15 plot results, show the MLflow leaderboard of best runs, re-check the submission file and download it.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Run the cells in order. Re-running section 2 recreates `assignment02.py` from scratch, so after editing any code cell re-run sections 2-9 in order.

**Submit:** rename `assignment02_submission.py` to `assignment02.py` and upload that `.py` file to CMS (never this `.ipynb`).

## 0. Install libraries and check the GPU

In [ ]:
!pip install -q torch torchvision datasets pillow mlflow
import torch
print('CUDA available:', torch.cuda.is_available(), '| Device:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Working directory

With `USE_DRIVE = True`, the code, the cached patch subset, `mlflow.db`, `mlruns/` and the submission file are kept on Google Drive, so they survive a Colab reset and MLflow can compare runs across sessions. If MLflow reports `database is locked` on Drive, set `USE_DRIVE = False`.

In [ ]:
import os, sys

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/came_a02'
else:
    WORK_DIR = '/content/came_a02'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('Working directory:', os.getcwd())

## 2. Module header, imports and constants

`TRAINED_MODEL_B64` stays empty here. The training script writes the filled-in copy to `assignment02_submission.py`.

`_EPOCH_CALLBACK` is an optional hook that only `main()` sets (for MLflow). It is always `None` on CMS.

In [ ]:
%%writefile assignment02.py
"""
CAME Assignment 02 - Colorectal Histopathology CNN Classifier (PyTorch).

Custom CNN trained FROM SCRATCH (random init, no pretrained weights) to classify
64x64 H&E colorectal tissue patches into 9 classes:
    ADI, BACK, DEB, LYM, MUC, MUS, NORM, STR, TUM

Colab workflow
--------------
1. Runtime -> Change runtime type -> T4 GPU.
2. Put this file in /content (upload it or use %%writefile assignment02.py).
3. !python assignment02.py
   (optional env vars: CAME_EPOCHS, CAME_BATCH_SIZE, CAME_PER_CLASS, CAME_HF_MODE)
4. main() trains, restores the best validation checkpoint, encodes it, verifies
   load_trained_model(cpu) and writes `assignment02_submission.py`: a copy of
   this file with TRAINED_MODEL_B64 already filled in. Download that file, rename
   it to assignment02.py and upload it to CMS.

Importing this module does NOT train and does NOT download data.
"""

import base64
import copy
import io
import os
import re
import time
import zlib
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder

# ---------------------------------------------------------------------------
# Base64 string of the Colab-trained best_model.pt (filled in after training).
# ---------------------------------------------------------------------------
TRAINED_MODEL_B64 = """
"""

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
NUM_CLASSES = 9
IMAGE_SIZE = 64
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)
SEED = 42
VAL_FRACTION = 0.15

HF_DATASET_ID = "1aurent/NCT-CRC-HE"
HF_SPLIT = "NCT_CRC_HE_100K"

# Optional per-epoch hook: callable(epoch, record, improved). Set only by main()
# (MLflow tracking on Colab); it stays None on CMS, so grading is unaffected.
_EPOCH_CALLBACK = None


## 3. Task 1 - Data preprocessing and loaders (15 marks)

- `make_transforms(True)`: Resize(64) -> HFlip -> VFlip -> ToTensor -> Normalize(0.5). `make_transforms(False)` has no flips.
- **CMS** (`CAME_DATA_DIR` set): `ImageFolder` on `train/` and `validation/` only. `datasets` is not imported and nothing is downloaded.
- **Colab**: streams `NCT_CRC_HE_100K` **one parquet shard at a time** (31 shards), so RAM stays low. It takes the first rows of each shard until every class has `CAME_PER_CLASS` patches (default 1000), caches the 64x64 subset to disk (~110 MB) and makes a per-class 85/15 split with seed 42. `CAME_HF_MODE=full` downloads the whole split instead.
- Colab-only extra training augmentation (random 90-degree rotation + mild colour jitter) is applied inside the dataset wrapper, so `make_transforms` matches the spec exactly. Disable it with `CAME_EXTRA_AUG=0`.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Task 1 - Data preprocessing & loaders
# ===========================================================================
def official_classes() -> list[str]:
    return ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]


def make_transforms(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(NORM_MEAN, NORM_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ])


class NCTPatchDataset(Dataset):
    """In-memory dataset of pre-resized 64x64 RGB patches (Colab / Hugging Face path).

    images: uint8 tensor of shape (N, 64, 64, 3); labels: long tensor of shape (N,).
    Each item is (tensor_3x64x64, int_label_0_to_8).
    """

    def __init__(self, images: torch.Tensor, labels: torch.Tensor, transform, extra_aug: bool = False):
        self.images = images
        self.labels = labels
        self.transform = transform
        self.extra_aug = extra_aug
        # Mild stain/colour variation to help generalise to unseen patients.
        self.color_jitter = transforms.ColorJitter(
            brightness=0.10, contrast=0.10, saturation=0.10, hue=0.02
        )

    def __len__(self) -> int:
        return int(self.labels.numel())

    def __getitem__(self, idx):
        from PIL import Image

        img = Image.fromarray(self.images[idx].numpy())  # (64, 64, 3) uint8 -> RGB
        if self.extra_aug:
            # Tissue has no canonical orientation -> random 90-degree rotations.
            k = int(torch.randint(0, 4, (1,)).item())
            if k:
                img = img.rotate(90 * k)
            img = self.color_jitter(img)
        x = self.transform(img)
        return x, int(self.labels[idx])


def _label_mapper(features):
    """Map Hugging Face labels (ClassLabel int or str) to official indices 0..8."""
    classes = official_classes()
    remap = None
    try:
        names = list(getattr(features["label"], "names", None) or [])
        if names:
            remap = {i: classes.index(n.strip().upper()) for i, n in enumerate(names)
                     if n.strip().upper() in classes}
    except Exception:
        remap = None

    def to_idx(value) -> int:
        if isinstance(value, str):
            return classes.index(value.strip().upper())
        value = int(value)
        if remap is not None:
            return remap.get(value, -1)
        return value

    return to_idx


def _patch_to_uint8(image_field) -> torch.Tensor:
    """Decode one HF image field (PIL or {'bytes', 'path'}) to a (64, 64, 3) uint8 tensor."""
    import numpy as np
    from PIL import Image

    if isinstance(image_field, dict):
        if image_field.get("bytes") is not None:
            img = Image.open(io.BytesIO(image_field["bytes"]))
        else:
            img = Image.open(image_field["path"])
    else:
        img = image_field
    img = img.convert("RGB")
    # Same resize as make_transforms(False) so train / test pixels match.
    img = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))(img)
    return torch.from_numpy(np.array(img, dtype=np.uint8).copy())


def _list_hf_shards() -> list[str]:
    """Return the parquet shard paths of the NCT_CRC_HE_100K split (relative to the repo)."""
    try:
        from huggingface_hub import HfFileSystem

        fs = HfFileSystem()
        files = sorted(fs.glob(f"datasets/{HF_DATASET_ID}/data/{HF_SPLIT}-*.parquet"))
        shards = ["data/" + f.rsplit("/", 1)[-1] for f in files]
        if shards:
            return shards
    except Exception as exc:  # pragma: no cover - network dependent
        print(f"[data] could not list shards via HfFileSystem ({exc}); using default pattern")
    return [f"data/{HF_SPLIT}-{i:05d}-of-00031-*.parquet" for i in range(31)]


def _collect_streaming(per_class: int):
    """Stream the NCT_CRC_HE_100K split ONE parquet shard at a time until every class
    has `per_class` patches. Avoids downloading the full ~15 GB split and keeps RAM
    bounded: only a single shard stream is open at any moment and it is freed before
    the next one is opened.

    Round r reads rows [r*R, (r+1)*R) from each of the 31 shards in turn. The shards
    are grouped by class, so round 0 already samples every class from several shards
    (= several slides); later rounds run only if a class is still short. The read
    order is fixed, so the subset is deterministic.
    """
    import gc

    from datasets import Image as HFImage
    from datasets import load_dataset

    shards = _list_hf_shards()
    default_rows = max(64, (per_class * NUM_CLASSES * 13) // (10 * len(shards)))
    rows_per_round = int(os.environ.get("CAME_ROWS_PER_SHARD", default_rows))
    max_rounds = int(os.environ.get("CAME_MAX_ROUNDS", 12))
    print(f"[data] streaming {HF_DATASET_ID}:{HF_SPLIT}: {len(shards)} shards, "
          f"{rows_per_round} rows per shard per round (one shard open at a time)")

    buckets = {c: [] for c in range(NUM_CLASSES)}
    exhausted = set()
    to_idx = None
    scanned = 0
    t0 = time.time()

    def all_full() -> bool:
        return all(len(buckets[c]) >= per_class for c in range(NUM_CLASSES))

    for rnd in range(max_rounds):
        if all_full() or len(exhausted) == len(shards):
            break
        for si, shard in enumerate(shards):
            if all_full():
                break
            if si in exhausted:
                continue
            ds = load_dataset(HF_DATASET_ID, data_files={HF_SPLIT: shard},
                              split=HF_SPLIT, streaming=True)
            try:
                ds = ds.cast_column("image", HFImage(decode=False))  # decode only what we keep
            except Exception:
                pass
            if to_idx is None:
                to_idx = _label_mapper(ds.features)
            if rnd:
                ds = ds.skip(rnd * rows_per_round)
            n_read = 0
            for example in ds.take(rows_per_round):
                n_read += 1
                y = to_idx(example["label"])
                if 0 <= y < NUM_CLASSES and len(buckets[y]) < per_class:
                    buckets[y].append(_patch_to_uint8(example["image"]))
            scanned += n_read
            if n_read < rows_per_round:
                exhausted.add(si)
            del ds
            gc.collect()
            counts = [len(buckets[c]) for c in range(NUM_CLASSES)]
            print(f"[data] round {rnd + 1} shard {si + 1}/{len(shards)} | scanned={scanned} "
                  f"per-class={counts} ({time.time() - t0:.0f}s)")
    return buckets


def _collect_full(per_class: int):
    """Download the full NCT_CRC_HE_100K split and draw a seeded balanced subset."""
    from datasets import Image as HFImage
    from datasets import load_dataset

    ds = load_dataset(HF_DATASET_ID, split=HF_SPLIT)
    to_idx = _label_mapper(ds.features)
    labels = torch.tensor([to_idx(v) for v in ds["label"]], dtype=torch.long)
    g = torch.Generator().manual_seed(SEED)
    chosen = []
    for c in range(NUM_CLASSES):
        idx = (labels == c).nonzero(as_tuple=False).view(-1)
        idx = idx[torch.randperm(idx.numel(), generator=g)][:per_class]
        chosen.extend(idx.tolist())
    chosen.sort()
    sub = ds.select(chosen)
    try:
        sub = sub.cast_column("image", HFImage(decode=False))
    except Exception:
        pass
    buckets = {c: [] for c in range(NUM_CLASSES)}
    for example in sub:
        y = to_idx(example["label"])
        buckets[y].append(_patch_to_uint8(example["image"]))
    return buckets


def _load_hf_subset(pool_per_class: int):
    """Return (images uint8 (N,64,64,3), labels long (N,)): a balanced pool of up to
    `pool_per_class` patches per class, cached on disk."""
    mode = os.environ.get("CAME_HF_MODE", "stream").lower()
    cache = Path(os.environ.get("CAME_CACHE", f"nct_crc_he_100k_subset_{pool_per_class}.pt"))

    if cache.exists():
        print(f"[data] loading cached subset from {cache}")
        blob = torch.load(cache, map_location="cpu", weights_only=True)
        return blob["images"], blob["labels"]

    if mode == "full":
        buckets = _collect_full(pool_per_class)
    else:
        buckets = _collect_streaming(pool_per_class)

    images, labels = [], []
    for c in range(NUM_CLASSES):
        if not buckets[c]:
            raise RuntimeError(f"No patches collected for class {official_classes()[c]}")
        images.extend(buckets[c])
        labels.extend([c] * len(buckets[c]))
    images = torch.stack(images).to(torch.uint8)
    labels = torch.tensor(labels, dtype=torch.long)
    try:
        torch.save({"images": images, "labels": labels}, cache)
        print(f"[data] cached subset to {cache}")
    except Exception as exc:
        print(f"[data] could not cache subset ({exc})")
    return images, labels


def _select_and_split(labels: torch.Tensor, per_class: int,
                      val_fraction: float = VAL_FRACTION, seed: int = SEED):
    """Deterministic balanced selection + per-class 85 / 15 split.

    Each class's pool is shuffled once with `seed` (independent of `per_class`); the
    first `per_class` items are used and the first 15 % of those are validation.
    So runs with different `per_class` on the same pool have nested validation sets,
    and the first round(0.15 * min_per_class) items of every class are validation
    (never training) in every run -> a fair common set for comparing sweep runs.
    Returns (train_idx, val_idx); val_idx is grouped by class in shuffled order.
    """
    g = torch.Generator().manual_seed(seed)
    train_idx, val_idx = [], []
    for c in range(NUM_CLASSES):
        idx = (labels == c).nonzero(as_tuple=False).view(-1)
        idx = idx[torch.randperm(idx.numel(), generator=g)][:per_class]
        n_val = int(round(idx.numel() * val_fraction))
        val_idx.append(idx[:n_val])
        train_idx.append(idx[n_val:])
    return torch.cat(train_idx), torch.cat(val_idx)


def _common_val_subset(val_dataset, n_per_class: int):
    """First `n_per_class` validation items of every class (identical across sweep runs)."""
    from torch.utils.data import Subset

    keep = []
    for c in range(NUM_CLASSES):
        keep.extend((val_dataset.labels == c).nonzero(as_tuple=False).view(-1)[:n_per_class].tolist())
    return Subset(val_dataset, keep)


def build_datasets():
    data_dir = os.environ.get("CAME_DATA_DIR")
    if data_dir:
        # CMS path: local ImageFolder only, no `datasets` import, no internet.
        train_dataset = ImageFolder(os.path.join(data_dir, "train"), transform=make_transforms(True))
        val_dataset = ImageFolder(os.path.join(data_dir, "validation"), transform=make_transforms(False))
        return train_dataset, val_dataset

    # Colab path: Hugging Face 1aurent/NCT-CRC-HE, split NCT_CRC_HE_100K.
    # CAME_POOL_PER_CLASS (>= CAME_PER_CLASS) lets sweep runs share one cached pool
    # and nested validation sets even when CAME_PER_CLASS differs between runs.
    per_class = int(os.environ.get("CAME_PER_CLASS", 1000))
    pool_per_class = max(per_class, int(os.environ.get("CAME_POOL_PER_CLASS", per_class)))
    images, labels = _load_hf_subset(pool_per_class)
    train_idx, val_idx = _select_and_split(labels, per_class)
    extra_aug = os.environ.get("CAME_EXTRA_AUG", "1") != "0"
    train_dataset = NCTPatchDataset(images[train_idx], labels[train_idx],
                                    make_transforms(True), extra_aug=extra_aug)
    val_dataset = NCTPatchDataset(images[val_idx], labels[val_idx],
                                  make_transforms(False), extra_aug=False)
    print(f"[data] train={len(train_dataset)} val={len(val_dataset)} "
          f"(per-class train counts={torch.bincount(labels[train_idx], minlength=NUM_CLASSES).tolist()})")
    return train_dataset, val_dataset


def make_loaders(train_dataset, val_dataset, batch_size: int):
    num_workers = int(os.environ.get("CAME_WORKERS", 0))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers)
    return train_loader, val_loader


## 4. Task 2 - Custom CNN from scratch (15 marks)

A plain `nn.Sequential` built only from allowed layers. Channels funnel 3 -> 32 -> 64 -> 128 -> 256 while the spatial size goes 64 -> 32 -> 16 -> 8 -> 4, with 6 x `Conv2d(3x3, padding=1)` + BatchNorm + ReLU and 2x2 MaxPool. The head is GlobalAvgPool -> Flatten -> Dropout(0.3) -> `Linear(256, 9)`, returning raw logits (no softmax). **437,481** trainable parameters.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Task 2 - Custom CNN (from scratch)
# ===========================================================================
def _conv_bn_relu(in_ch: int, out_ch: int) -> list:
    return [
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    ]


def build_model(num_classes: int = 9) -> torch.nn.Module:
    """Funnel CNN 3 -> 32 -> 64 -> 128 -> 256 channels, 64 -> 4 spatial.

    Uses only allowed layers (Sequential, Conv2d, BatchNorm2d, ReLU, MaxPool2d,
    AdaptiveAvgPool2d, Flatten, Dropout, Linear). Returns raw logits (N, num_classes).
    Trainable parameters (num_classes=9): 437,481.

    Dropout p is 0.3 unless CAME_DROPOUT is set (Colab sweeps only). Dropout has no
    weights and is inactive in eval mode, so it never affects loading or predictions.
    """
    dropout_p = float(os.environ.get("CAME_DROPOUT", 0.3))
    features = nn.Sequential(
        # Block 1: 3 -> 32, 64x64 -> 32x32
        nn.Sequential(*_conv_bn_relu(3, 32), *_conv_bn_relu(32, 32), nn.MaxPool2d(2, 2)),
        # Block 2: 32 -> 64, 32x32 -> 16x16
        nn.Sequential(*_conv_bn_relu(32, 64), *_conv_bn_relu(64, 64), nn.MaxPool2d(2, 2)),
        # Block 3: 64 -> 128, 16x16 -> 8x8
        nn.Sequential(*_conv_bn_relu(64, 128), nn.MaxPool2d(2, 2)),
        # Block 4: 128 -> 256, 8x8 -> 4x4
        nn.Sequential(*_conv_bn_relu(128, 256), nn.MaxPool2d(2, 2)),
    )
    classifier = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Dropout(p=dropout_p),
        nn.Linear(256, num_classes),
    )
    return nn.Sequential(features, classifier)


def count_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 5. Task 3 - Training loop, validation and best checkpoint restoration (25 marks)

Follows the spec exactly: `torch.manual_seed(42)`, `CrossEntropyLoss`, `AdamW(lr=1e-3, weight_decay=1e-4)`, sample-weighted loss and accuracy, and a deep copy of the `state_dict` whenever `val_acc` strictly improves (ties within 1e-12 broken by lower `val_loss`). The best state is restored after the loop and `history` holds the four required keys per epoch.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Task 3 - Training loop, validation & best checkpoint restoration
# ===========================================================================
def train_model(train_loader, val_loader, epochs: int, device: torch.device):
    torch.manual_seed(42)
    model = build_model(9).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0
    best_val_loss = float("inf")
    history = []

    for epoch in range(epochs):
        t0 = time.time()

        # ---- training phase ----
        model.train()
        loss_sum, correct, seen = 0.0, 0, 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device).long()
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            bs = labels.size(0)
            loss_sum += loss.item() * bs
            correct += (logits.argmax(dim=1) == labels).sum().item()
            seen += bs
        train_loss = loss_sum / max(seen, 1)
        train_acc = correct / max(seen, 1)

        # ---- validation phase ----
        model.eval()
        loss_sum, correct, seen = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device).long()
                logits = model(images)
                loss = criterion(logits, labels)
                bs = labels.size(0)
                loss_sum += loss.item() * bs
                correct += (logits.argmax(dim=1) == labels).sum().item()
                seen += bs
        val_loss = loss_sum / max(seen, 1)
        val_acc = correct / max(seen, 1)

        # ---- checkpoint tracking ----
        if abs(val_acc - best_val_acc) <= 1e-12:
            improved = val_loss < best_val_loss
        else:
            improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        history.append({
            "train_loss": float(train_loss),
            "train_acc": float(train_acc),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
        })
        print(f"Epoch {epoch + 1:02d}/{epochs} | train_loss={train_loss:.4f} "
              f"train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
              f"{' *best*' if improved else ''} | {time.time() - t0:.1f}s")
        if _EPOCH_CALLBACK is not None:
            try:
                _EPOCH_CALLBACK(epoch, history[-1], improved)
            except Exception as exc:  # tracking must never break training
                print(f"[tracking] epoch callback failed: {exc}")

    model.load_state_dict(best_state)
    return model, history


## 6. Task 4 - Inference and evaluation metrics (10 marks)

`predict` returns a 1-D `torch.long` CPU tensor in loader order. `classification_report` returns `accuracy`, `macro_recall` (over classes present in `y_true`) and a `(9, 9)` long `confusion` matrix (rows = true, cols = predicted).

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Task 4 - Inference & metrics
# ===========================================================================
def predict(model: torch.nn.Module, loader, device: torch.device) -> torch.Tensor:
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            images = batch[0] if isinstance(batch, (list, tuple)) else batch
            images = images.to(device)
            logits = model(images)
            preds.append(logits.argmax(dim=1).detach().cpu().long())
    if not preds:
        return torch.empty(0, dtype=torch.long)
    return torch.cat(preds).long()


def classification_report(y_true: torch.Tensor, y_pred: torch.Tensor) -> dict:
    n = NUM_CLASSES
    y_true = torch.as_tensor(y_true).detach().cpu().long().view(-1)
    y_pred = torch.as_tensor(y_pred).detach().cpu().long().view(-1)
    if y_true.numel() != y_pred.numel():
        raise ValueError(f"y_true has {y_true.numel()} items but y_pred has {y_pred.numel()}")

    confusion = torch.zeros((n, n), dtype=torch.long)
    if y_true.numel() > 0:
        valid = (y_true >= 0) & (y_true < n) & (y_pred >= 0) & (y_pred < n)
        flat = y_true[valid] * n + y_pred[valid]
        confusion = torch.bincount(flat, minlength=n * n).reshape(n, n).long()
        accuracy = (y_true == y_pred).double().mean().item()
    else:
        accuracy = 0.0

    support = confusion.sum(dim=1)
    present = support > 0
    if present.any():
        recalls = confusion.diag()[present].double() / support[present].double()
        macro_recall = recalls.mean().item()
    else:
        macro_recall = 0.0

    return {
        "accuracy": float(accuracy),
        "macro_recall": float(macro_recall),
        "confusion": confusion,
    }


## 7. Task 5 - Embedding the Colab-trained weights (35 marks)

`encode_model_to_b64` and `load_trained_model` are the provided helper code. The only addition is a clear error if `TRAINED_MODEL_B64` is empty. `_write_submission_file` copies the module with the base64 weights filled in, wrapped at 100 characters; `b64decode` ignores the newlines.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Task 5 - Embedding the Colab-trained model
# ===========================================================================
def encode_model_to_b64(model: torch.nn.Module) -> str:
    """Run this on Colab after training and paste the output into TRAINED_MODEL_B64."""
    state = {k: v.detach().cpu().half() for k, v in model.state_dict().items()}
    buf = io.BytesIO()
    torch.save(state, buf)
    compressed = zlib.compress(buf.getvalue(), level=9)
    return base64.b64encode(compressed).decode("ascii")


def load_trained_model(device: torch.device) -> torch.nn.Module:
    """Rebuilds build_model(9) and loads TRAINED_MODEL_B64 on CMS."""
    if not TRAINED_MODEL_B64.strip():
        raise RuntimeError("TRAINED_MODEL_B64 is empty - train on Colab and paste the encoded weights.")
    model = build_model(9)
    raw = zlib.decompress(base64.b64decode(TRAINED_MODEL_B64.strip()))
    state = torch.load(io.BytesIO(raw), map_location="cpu", weights_only=True)
    state = {k: v.float() if v.is_floating_point() else v for k, v in state.items()}
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model


def _write_submission_file(b64: str, out_path: str = "assignment02_submission.py"):
    """Copy this source file with TRAINED_MODEL_B64 filled in (wrapped at 100 chars;
    base64.b64decode ignores the newlines). Returns the written path or None."""
    try:
        source = Path(__file__).read_text(encoding="utf-8")
    except Exception as exc:
        print(f"[export] could not read source file ({exc}); paste trained_model_b64.txt manually")
        return None
    wrapped = "\n".join(b64[i:i + 100] for i in range(0, len(b64), 100))
    new_source, n_subs = re.subn(
        r'^TRAINED_MODEL_B64 = """.*?"""',
        lambda _m: 'TRAINED_MODEL_B64 = """\n' + wrapped + '\n"""',
        source, count=1, flags=re.DOTALL | re.MULTILINE,
    )
    if n_subs != 1:
        print("[export] TRAINED_MODEL_B64 placeholder not found; paste trained_model_b64.txt manually")
        return None
    Path(out_path).write_text(new_source, encoding="utf-8")
    print(f"[export] wrote {out_path} ({len(new_source) / 1e6:.2f} MB)")
    return out_path


## 8. MLflow experiment tracking (Colab only)

`mlflow` is imported only inside these helpers, which only `main()` calls, so CMS never needs it. Each run logs its settings, live per-epoch metrics, best-so-far metrics, per-class recall, the CPU fp16 reload check and the files `best_model.pt`, `assignment02_submission.py`, `history.json` and `confusion_matrix.json`. Runs are ranked by `best_val_macro_recall`.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# MLflow experiment tracking (Colab only - never imported on CMS)
# ===========================================================================
MLFLOW_EXPERIMENT = os.environ.get("CAME_MLFLOW_EXPERIMENT", "came-a02-nct-crc-cnn")
MLFLOW_RANK_METRIC = os.environ.get("CAME_RANK_METRIC", "best_val_macro_recall")


def _mlflow_start(params: dict):
    """Start an MLflow run and log params. Returns the mlflow module, or None if
    tracking is disabled (CAME_MLFLOW=0) or mlflow is not installed."""
    if os.environ.get("CAME_MLFLOW", "1") == "0":
        return None
    try:
        import mlflow
    except ImportError:
        print("[mlflow] not installed (pip install mlflow) - continuing without tracking")
        return None
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db"))
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    run = mlflow.start_run(run_name=os.environ.get("CAME_RUN_NAME"))
    mlflow.log_params(params)
    tags = {"task": "NCT-CRC-HE 9-class", "framework": "pytorch", "pretrained": "no"}
    if os.environ.get("CAME_SWEEP_ID"):
        tags["sweep_id"] = os.environ["CAME_SWEEP_ID"]
    mlflow.set_tags(tags)
    print(f"[mlflow] run {run.info.run_id} in experiment '{MLFLOW_EXPERIMENT}' "
          f"({mlflow.get_tracking_uri()})")
    return mlflow


def _mlflow_epoch_callback(mlflow):
    """Live per-epoch logging + running best-so-far metrics."""
    def callback(epoch: int, record: dict, improved: bool) -> None:
        mlflow.log_metrics(record, step=epoch + 1)
        if improved:
            mlflow.log_metrics({
                "best_val_acc": record["val_acc"],
                "best_val_loss": record["val_loss"],
                "best_epoch": epoch + 1,
            }, step=epoch + 1)
    return callback


def _mlflow_log_results(mlflow, report: dict, cpu_report: dict, history: list,
                        train_seconds: float, artifacts: list, extra_metrics=None) -> None:
    classes = official_classes()
    conf = report["confusion"]
    support = conf.sum(dim=1).clamp(min=1).double()
    per_class = (conf.diag().double() / support).tolist()

    metrics = {
        "best_val_accuracy": report["accuracy"],
        "best_val_macro_recall": report["macro_recall"],
        "cpu_fp16_val_accuracy": cpu_report["accuracy"],
        "cpu_fp16_val_macro_recall": cpu_report["macro_recall"],
        "train_seconds": train_seconds,
    }
    metrics.update({f"recall_{name}": r for name, r in zip(classes, per_class)})
    metrics.update(extra_metrics or {})
    mlflow.log_metrics(metrics)
    mlflow.set_tag("cms_target_met", str(cpu_report["macro_recall"] >= 0.80))

    mlflow.log_dict(history, "history.json")
    mlflow.log_dict({
        "classes": classes,
        "confusion": conf.tolist(),
        "per_class_recall": dict(zip(classes, per_class)),
    }, "confusion_matrix.json")
    for path in artifacts:
        if path and Path(path).exists():
            mlflow.log_artifact(path)


def _mlflow_print_leaderboard(mlflow, top_k: int = 5) -> None:
    """Print the best runs so far, ranked by validation macro recall."""
    try:
        runs = mlflow.search_runs(
            experiment_names=[MLFLOW_EXPERIMENT],
            order_by=[f"metrics.{MLFLOW_RANK_METRIC} DESC"],
            max_results=top_k,
        )
    except Exception as exc:
        print(f"[mlflow] could not query runs: {exc}")
        return
    if runs is None or len(runs) == 0:
        return
    wanted = ["run_id", "tags.mlflow.runName", f"metrics.{MLFLOW_RANK_METRIC}",
              "metrics.common_val_macro_recall", "metrics.best_val_macro_recall",
              "metrics.best_val_accuracy", "metrics.best_epoch", "params.epochs",
              "params.per_class", "params.batch_size", "params.extra_aug", "params.dropout"]
    wanted = list(dict.fromkeys(wanted))  # drop duplicates, keep order
    cols = [c for c in wanted if c in runs.columns]
    print(f"\n[mlflow] Top {len(runs)} runs by {MLFLOW_RANK_METRIC}:")
    print(runs[cols].to_string(index=False))
    best = runs.iloc[0]
    print(f"[mlflow] best run: {best['run_id']} -> download its "
          f"assignment02_submission.py artifact to submit that model\n")


## 9. Colab entry point and the required CMS ending

`main()` only runs under `if __name__ == "__main__":` (or when called explicitly below), so importing the module never trains or downloads. `_run_on_cms_if_needed()` and the `__main__` block are kept exactly as the assignment requires.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Colab entry point
# ===========================================================================
def main():
    global TRAINED_MODEL_B64, _EPOCH_CALLBACK

    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("CUDA available:", torch.cuda.is_available(), "| Device:",
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

    batch_size = int(os.environ.get("CAME_BATCH_SIZE", 64 if torch.cuda.is_available() else 32))
    epochs = int(os.environ.get("CAME_EPOCHS", 12))
    per_class = int(os.environ.get("CAME_PER_CLASS", 1000))
    pool_per_class = max(per_class, int(os.environ.get("CAME_POOL_PER_CLASS", per_class)))
    common_n = int(os.environ.get("CAME_COMMON_VAL_PER_CLASS", 0))

    train_dataset, val_dataset = build_datasets()
    train_loader, val_loader = make_loaders(train_dataset, val_dataset, batch_size)

    mlflow = _mlflow_start({
        "epochs": epochs,
        "batch_size": batch_size,
        "per_class": per_class,
        "pool_per_class": pool_per_class,
        "common_val_per_class": common_n,
        "dropout": float(os.environ.get("CAME_DROPOUT", 0.3)),
        "hf_mode": os.environ.get("CAME_HF_MODE", "stream"),
        "extra_aug": os.environ.get("CAME_EXTRA_AUG", "1") != "0",
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
        "optimizer": "AdamW",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "seed": SEED,
        "image_size": IMAGE_SIZE,
        "num_params": count_parameters(build_model(9)),
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    })
    if mlflow is not None:
        _EPOCH_CALLBACK = _mlflow_epoch_callback(mlflow)

    try:
        print(f"Model parameters: {count_parameters(build_model(9)):,}")
        print(f"Training for {epochs} epochs, batch size {batch_size}")
        t0 = time.time()
        model, history = train_model(train_loader, val_loader, epochs, device)
        train_seconds = time.time() - t0
        print(f"Training finished in {train_seconds:.1f}s")

        y_true = torch.cat([labels.long() for _, labels in val_loader])
        y_pred = predict(model, val_loader, device)
        report = classification_report(y_true, y_pred)
        print(f"Best checkpoint (GPU) | val accuracy={report['accuracy']:.4f} "
              f"macro_recall={report['macro_recall']:.4f}")
        print("Confusion matrix (rows=true, cols=pred):", official_classes())
        print(report["confusion"])

        # Fixed common validation subset: identical for every sweep run -> fair ranking.
        extra_metrics, common_report = {}, None
        if common_n > 0 and hasattr(val_dataset, "labels"):
            common_ds = _common_val_subset(val_dataset, common_n)
            common_loader = DataLoader(common_ds, batch_size=batch_size, shuffle=False)
            common_true = torch.cat([labels.long() for _, labels in common_loader])
            common_report = classification_report(common_true, predict(model, common_loader, device))
            extra_metrics = {
                "common_val_accuracy": common_report["accuracy"],
                "common_val_macro_recall": common_report["macro_recall"],
            }
            print(f"Common validation set ({common_n}/class, same for all sweep runs) | "
                  f"accuracy={common_report['accuracy']:.4f} "
                  f"macro_recall={common_report['macro_recall']:.4f}")

        torch.save(model.state_dict(), "best_model.pt")
        b64 = encode_model_to_b64(model)
        Path("trained_model_b64.txt").write_text(b64, encoding="ascii")
        print(f"Saved best_model.pt and trained_model_b64.txt ({len(b64) / 1e6:.2f} MB of base64)")

        # Verify the exact CMS loading path on CPU (fp16 round trip).
        TRAINED_MODEL_B64 = b64
        cpu = torch.device("cpu")
        cpu_model = load_trained_model(cpu)
        cpu_report = classification_report(y_true, predict(cpu_model, val_loader, cpu))
        print(f"load_trained_model(cpu) OK | val accuracy={cpu_report['accuracy']:.4f} "
              f"macro_recall={cpu_report['macro_recall']:.4f}")

        submission = _write_submission_file(b64)

        if mlflow is not None:
            _mlflow_log_results(mlflow, report, cpu_report, history, train_seconds,
                                ["best_model.pt", submission], extra_metrics)
            mlflow.end_run()
            _mlflow_print_leaderboard(mlflow)

        # Returned for notebook use (plots / checks); ignored when run as a script.
        return {
            "model": model,
            "history": history,
            "report": report,
            "cpu_report": cpu_report,
            "common_report": common_report,
            "val_loader": val_loader,
            "submission": submission,
        }
    except BaseException:
        if mlflow is not None:
            mlflow.end_run(status="FAILED")
        raise
    finally:
        _EPOCH_CALLBACK = None


def _run_on_cms_if_needed() -> bool:
    zpath = Path("cms_eval_data.zip")
    if not zpath.exists():
        return False
    import sys, runpy, zipfile
    sys.modules["a02_cnn"] = sys.modules["__main__"]
    with zipfile.ZipFile(zpath) as archive:
        archive.extract("cms_grader.py", ".")
    runpy.run_path("cms_grader.py", run_name="__main__")
    return True


if __name__ == "__main__":
    if not _run_on_cms_if_needed():
        main()


## 10. Import the module and run the CMS gate checks locally

Importing must not train or download anything. This cell checks the 0-mark gate items and runs small unit tests of the metrics.

In [ ]:
import importlib
import torch
import torch.nn as nn
import assignment02 as a02
a02 = importlib.reload(a02)  # pick up any re-written cells

model = a02.build_model(9)
allowed = (nn.Sequential, nn.Conv2d, nn.BatchNorm2d, nn.ReLU, nn.MaxPool2d,
           nn.AdaptiveAvgPool2d, nn.Flatten, nn.Dropout, nn.Linear)
n_conv = sum(isinstance(m, nn.Conv2d) for m in model.modules())
bad = sorted({type(m).__name__ for m in model.modules() if not isinstance(m, allowed)})
n_params = a02.count_parameters(model)
with torch.no_grad():
    logits = model.eval()(torch.randn(4, 3, 64, 64))

print('official_classes :', a02.official_classes())
print('Conv2d layers    :', n_conv)
print('disallowed layers:', bad or 'none')
print(f'parameters       : {n_params:,}')
print('logits shape     :', tuple(logits.shape))
print('train transforms :', a02.make_transforms(True))
print('eval transforms  :', a02.make_transforms(False))

assert a02.official_classes() == ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
assert n_conv >= 3 and not bad
assert 25_000 <= n_params <= 500_000
assert logits.shape == (4, 9)

# Metrics sanity check: acc = 3/4, macro recall = (1 + 1 + 0.5) / 3
rep = a02.classification_report(torch.tensor([0, 1, 2, 2]), torch.tensor([0, 1, 2, 1]))
assert abs(rep['accuracy'] - 0.75) < 1e-9
assert abs(rep['macro_recall'] - 2.5 / 3) < 1e-9
assert rep['confusion'].shape == (9, 9) and rep['confusion'].dtype == torch.long
print('\nAll gate checks passed.')

## 11. Configure the training run

Ideas to compare in MLflow: `CAME_EPOCHS` 12 / 20 / 25 and `CAME_PER_CLASS` 1000 / 1500. A different `CAME_PER_CLASS` builds a new cached subset. The best validation checkpoint is always kept, so extra epochs can't make the saved model worse. Aim for validation macro recall of at least 0.90, because unseen patients usually score lower.

In [ ]:
import os

# Clear sweep-only settings so a single run behaves normally (see section 12b for sweeps).
for k in ['CAME_SWEEP_ID', 'CAME_RANK_METRIC', 'CAME_POOL_PER_CLASS', 'CAME_COMMON_VAL_PER_CLASS']:
    os.environ.pop(k, None)

os.environ['CAME_EPOCHS'] = '12'
os.environ['CAME_BATCH_SIZE'] = '64'
os.environ['CAME_PER_CLASS'] = '1000'
os.environ['CAME_DROPOUT'] = '0.3'
os.environ['CAME_HF_MODE'] = 'stream'   # or 'full' (downloads the whole ~15 GB split)
os.environ['CAME_EXTRA_AUG'] = '1'      # '0' = spec transforms only
os.environ['CAME_WORKERS'] = '2'
os.environ['CAME_MLFLOW'] = '1'
os.environ['CAME_RUN_NAME'] = (f"e{os.environ['CAME_EPOCHS']}_pc{os.environ['CAME_PER_CLASS']}"
                               f"_aug{os.environ['CAME_EXTRA_AUG']}_do{os.environ['CAME_DROPOUT']}")
print('Run name:', os.environ['CAME_RUN_NAME'])

## 12. Train on the GPU

This calls `main()`, which loads the data, trains, restores the best checkpoint, evaluates it, saves `best_model.pt` and `trained_model_b64.txt`, checks `load_trained_model(cpu)`, writes `assignment02_submission.py` and logs everything to MLflow.

The same thing can also be run as a script: `!python assignment02.py`.

In [ ]:
a02 = importlib.reload(a02)  # re-read env-dependent constants (e.g. experiment name)
result = a02.main()

## 12b. Hyperparameter sweep with MLflow (optional, instead of or after section 12)

Trains one model per combination of values in `SEARCH_SPACE`. Every run is an MLflow run tagged `sweep_id`, and the best one is chosen automatically.

**What is swept, and what is not.** The assignment fixes `torch.manual_seed(42)`, `AdamW(lr=1e-3, weight_decay=1e-4)` and `CrossEntropyLoss` inside `train_model()`. Stage A (65 marks) checks this, so those stay unchanged. The sweep varies only settings that are safe for CMS:
- `CAME_PER_CLASS`: training data size
- `CAME_BATCH_SIZE`
- `CAME_EXTRA_AUG`: rotation + colour jitter on/off
- `CAME_DROPOUT`: dropout has no weights and is off in eval mode, so it never changes how CMS loads or runs the model

**Fair comparison.** All runs share one cached patch pool (`CAME_POOL_PER_CLASS` = largest `CAME_PER_CLASS`). Each run is scored on a **common validation set**: the same `round(0.15 x smallest CAME_PER_CLASS)` patches per class, which no run ever trains on. Runs are ranked by `common_val_macro_recall`.

**Time.** About 3-6 minutes per run at 25 epochs on a T4, so `MAX_RUNS = 8` takes very roughly 30-45 minutes. The first run also downloads a larger patch pool. If Colab disconnects, re-run sections 0-10 and then these cells with the same `SWEEP_ID`: runs that already finished are skipped.

In [ ]:
import itertools
import random

# ---- Search space: each combination becomes one MLflow run ----
SEARCH_SPACE = {
    'CAME_PER_CLASS':  [1000, 1500, 2000],  # patches per class used for training + validation
    'CAME_BATCH_SIZE': [32, 64, 128],
    'CAME_EXTRA_AUG':  [1, 0],              # 90-degree rotations + colour jitter on / off
    'CAME_DROPOUT':    [0.2, 0.3, 0.5],
}
# Shared by every run. Epochs are not swept: the best-validation checkpoint is kept,
# so one long run already contains every shorter run (same seed, constant LR).
FIXED = {'CAME_EPOCHS': 25, 'CAME_HF_MODE': 'stream', 'CAME_WORKERS': 2}
BASELINE = {'CAME_PER_CLASS': 1000, 'CAME_BATCH_SIZE': 64, 'CAME_EXTRA_AUG': 1, 'CAME_DROPOUT': 0.3}
MAX_RUNS = 8           # grid has 54 combos -> baseline + 7 random ones (None = full grid)
SWEEP_ID = 'sweep_01'  # keep the same id to resume after a disconnect; change it for a new sweep

keys = list(SEARCH_SPACE)
grid = [dict(zip(keys, values)) for values in itertools.product(*SEARCH_SPACE.values())]
if MAX_RUNS and len(grid) > MAX_RUNS:
    rest = [cfg for cfg in grid if cfg != BASELINE]
    picked = random.Random(42).sample(rest, MAX_RUNS - int(BASELINE in grid))
    configs = ([BASELINE] if BASELINE in grid else []) + picked
else:
    configs = grid


def sweep_run_name(cfg):
    return (f"{SWEEP_ID}_pc{cfg['CAME_PER_CLASS']}_bs{cfg['CAME_BATCH_SIZE']}"
            f"_aug{cfg['CAME_EXTRA_AUG']}_do{cfg['CAME_DROPOUT']}")


print(f'{len(grid)} combinations in the grid -> {len(configs)} runs in {SWEEP_ID}:')
for cfg in configs:
    print('  ', sweep_run_name(cfg))

In [ ]:
import gc
import importlib
import os

import mlflow
import torch

# Settings shared by every run of the sweep
os.environ['CAME_SWEEP_ID'] = SWEEP_ID
os.environ['CAME_RANK_METRIC'] = 'common_val_macro_recall'
os.environ['CAME_POOL_PER_CLASS'] = str(max(SEARCH_SPACE['CAME_PER_CLASS']))
os.environ['CAME_COMMON_VAL_PER_CLASS'] = str(int(round(min(SEARCH_SPACE['CAME_PER_CLASS']) * 0.15)))
for k, v in FIXED.items():
    os.environ[k] = str(v)

# Runs of this sweep that already finished (lets you resume after a Colab disconnect)
mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI', 'sqlite:///mlflow.db'))
done = set()
try:
    prev = mlflow.search_runs(
        experiment_names=[a02.MLFLOW_EXPERIMENT],
        filter_string=f"tags.sweep_id = '{SWEEP_ID}' and attributes.status = 'FINISHED'")
    if len(prev):
        done = set(prev['tags.mlflow.runName'])
except Exception as exc:
    print('Could not read earlier runs (first sweep?):', exc)

for i, cfg in enumerate(configs, 1):
    name = sweep_run_name(cfg)
    if name in done:
        print(f'[{i}/{len(configs)}] {name}: already finished - skipped')
        continue
    for k, v in cfg.items():
        os.environ[k] = str(v)
    os.environ['CAME_RUN_NAME'] = name
    print(f"\n{'=' * 80}\n[{i}/{len(configs)}] {name}\n{'=' * 80}")
    a02 = importlib.reload(a02)  # re-read env-dependent constants
    res = a02.main()
    del res
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\nSweep finished.')

### Sweep results - pick the best run

The table and chart show every finished run of this sweep, ranked by `common_val_macro_recall`. The best run's `assignment02_submission.py` is downloaded from MLflow, and section 15 then checks and downloads exactly that file.

In [ ]:
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

RANK = 'metrics.common_val_macro_recall'
runs = mlflow.search_runs(
    experiment_names=[a02.MLFLOW_EXPERIMENT],
    filter_string=f"tags.sweep_id = '{SWEEP_ID}' and attributes.status = 'FINISHED'",
    order_by=[f'{RANK} DESC', 'metrics.best_val_macro_recall DESC'],
)
cols = ['tags.mlflow.runName', RANK, 'metrics.common_val_accuracy', 'metrics.best_val_macro_recall',
        'metrics.cpu_fp16_val_macro_recall', 'metrics.best_epoch', 'metrics.train_seconds',
        'params.per_class', 'params.batch_size', 'params.extra_aug', 'params.dropout', 'run_id']
display(runs[[c for c in cols if c in runs.columns]])

# One bar per run, best at the top
plot_df = runs.iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 0.45 * len(plot_df) + 1.5))
ax.barh(plot_df['tags.mlflow.runName'], plot_df[RANK], color='tab:blue')
ax.axvline(0.80, color='gray', linestyle='--', label='CMS 35/35 threshold (0.80)')
ax.set_xlim(max(0.0, plot_df[RANK].min() - 0.05), 1.0)
ax.set_xlabel('macro recall on the common validation set')
ax.set_title(f'{SWEEP_ID}: runs ranked by common validation macro recall')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

best = runs.iloc[0]
print('Best run :', best['tags.mlflow.runName'], '| run_id', best['run_id'])
print(f"common_val_macro_recall = {best[RANK]:.4f}")
print('Best settings:', {k: best[f'params.{k}'] for k in
                         ['per_class', 'batch_size', 'extra_aug', 'dropout', 'epochs']})

best_submission_path = mlflow.artifacts.download_artifacts(
    run_id=best['run_id'], artifact_path='assignment02_submission.py',
    dst_path=f'best_run_{SWEEP_ID}')
print('Best submission file:', best_submission_path)

# Loader for section 15 = the common validation set (no sweep run trained on it).
os.environ['CAME_PER_CLASS'] = str(min(SEARCH_SPACE['CAME_PER_CLASS']))
_, _common_val = a02.build_datasets()
VERIFY_LOADER = DataLoader(_common_val, batch_size=128, shuffle=False)
print('VERIFY_LOADER (common validation set) size:', len(_common_val))

## 13. Learning curves and confusion matrix

In [ ]:
import matplotlib.pyplot as plt

h = result['history']
epochs_axis = range(1, len(h) + 1)

# Same best-epoch rule as train_model()
best_i, best_acc, best_loss = 0, -1.0, float('inf')
for i, r in enumerate(h):
    better = (r['val_loss'] < best_loss) if abs(r['val_acc'] - best_acc) <= 1e-12 else (r['val_acc'] > best_acc)
    if better:
        best_i, best_acc, best_loss = i, r['val_acc'], r['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in [(axes[0], 'loss', 'Cross-entropy loss'), (axes[1], 'acc', 'Accuracy')]:
    ax.plot(epochs_axis, [r[f'train_{key}'] for r in h], marker='o', label='train')
    ax.plot(epochs_axis, [r[f'val_{key}'] for r in h], marker='o', label='validation')
    ax.axvline(best_i + 1, color='gray', linestyle='--', label=f'best epoch ({best_i + 1})')
    ax.set_title(title)
    ax.set_xlabel('epoch')
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()
print(f"Best epoch {best_i + 1}: val_acc={best_acc:.4f}, val_loss={best_loss:.4f}")

In [ ]:
import numpy as np

classes = a02.official_classes()
conf = result['report']['confusion'].numpy()
norm = conf / np.clip(conf.sum(axis=1, keepdims=True), 1, None)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(classes)), classes, rotation=45)
ax.set_yticks(range(len(classes)), classes)
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, int(conf[i, j]), ha='center', va='center', fontsize=8,
                color='white' if norm[i, j] > 0.5 else 'black')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f"Validation confusion matrix | acc={result['report']['accuracy']:.3f} "
             f"macro recall={result['report']['macro_recall']:.3f}")
fig.colorbar(im, ax=ax, label='row-normalised (recall)')
plt.tight_layout()
plt.show()

for name, r in zip(classes, norm.diagonal()):
    print(f'{name:5s} recall = {r:.3f}')
print(f"CPU fp16 reload: acc={result['cpu_report']['accuracy']:.4f} "
      f"macro recall={result['cpu_report']['macro_recall']:.4f}")

## 14. MLflow - best runs so far

Ranks every run in the experiment by `best_val_macro_recall` and downloads the **best run's** `assignment02_submission.py` into `best_run/`. That may be a run from an earlier session if you work on Drive.

In [ ]:
import mlflow

mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI', 'sqlite:///mlflow.db'))
runs = mlflow.search_runs(
    experiment_names=[a02.MLFLOW_EXPERIMENT],
    order_by=[f'metrics.{a02.MLFLOW_RANK_METRIC} DESC'],
)
wanted = ['run_id', 'tags.mlflow.runName', 'tags.sweep_id', 'status',
          f'metrics.{a02.MLFLOW_RANK_METRIC}', 'metrics.common_val_macro_recall',
          'metrics.best_val_macro_recall', 'metrics.best_val_accuracy',
          'metrics.cpu_fp16_val_macro_recall', 'metrics.best_epoch', 'metrics.train_seconds',
          'params.epochs', 'params.per_class', 'params.batch_size', 'params.extra_aug',
          'params.dropout', 'tags.cms_target_met']
wanted = list(dict.fromkeys(wanted))  # drop duplicates, keep order
display(runs[[c for c in wanted if c in runs.columns]].head(10))

# If you ran a sweep (12b), its best submission file is already in best_submission_path.
if 'best_submission_path' not in globals():
    finished = runs[runs['status'] == 'FINISHED'] if 'status' in runs.columns else runs
    best_run_id = finished.iloc[0]['run_id']
    best_submission_path = mlflow.artifacts.download_artifacts(
        run_id=best_run_id, artifact_path='assignment02_submission.py', dst_path='best_run')
    print('Best run:', best_run_id)
print('Best submission file:', best_submission_path)

Optional: open the MLflow UI inside the notebook output (an iframe). The leaderboard above already works without it.

In [ ]:
import os
import socket
import subprocess
import time

PORT = 5000

def _port_open(port: int) -> bool:
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

# Stop any MLflow UI/server started earlier (it may be running without the host settings).
subprocess.run(['pkill', '-f', 'mlflow (ui|server)'], check=False)
subprocess.run(['fuser', '-k', f'{PORT}/tcp'], check=False, capture_output=True)
for _ in range(20):
    if not _port_open(PORT):
        break
    time.sleep(0.5)

# MLflow's security middleware rejects Colab's proxy hostname ("Invalid Host header").
# Allow it: this server only listens on 127.0.0.1 inside your own Colab VM.
env = dict(os.environ,
           MLFLOW_SERVER_ALLOWED_HOSTS='*',
           MLFLOW_SERVER_CORS_ALLOWED_ORIGINS='*')
cmd = ['mlflow', 'server', '--backend-store-uri', 'sqlite:///mlflow.db',
       '--host', '127.0.0.1', '--port', str(PORT)]
help_text = subprocess.run(['mlflow', 'server', '--help'], capture_output=True, text=True).stdout
if '--allowed-hosts' in help_text:
    cmd += ['--allowed-hosts', '*']
if '--cors-allowed-origins' in help_text:
    cmd += ['--cors-allowed-origins', '*']

ui_proc = subprocess.Popen(cmd, env=env, stdout=open('mlflow_ui.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(90):  # wait up to ~45 s for the server to come up
    if _port_open(PORT):
        break
    time.sleep(0.5)
else:
    print('MLflow server did not start - run: !tail -n 30 mlflow_ui.log')

from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=800)

## 15. Verify and download the submission file

This loads the submission file as a fresh module, rebuilds the model from its embedded `TRAINED_MODEL_B64` on the CPU (exactly as CMS does) and scores it on the validation split. It then downloads the file. **Rename it to `assignment02.py` before uploading to CMS.**

In [ ]:
import importlib.util

submission_path = globals().get('best_submission_path') or result['submission']
spec = importlib.util.spec_from_file_location('a02_submission', submission_path)
sub = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sub)  # import only: must not train or download

# After a sweep, check on the common validation set (no sweep run trained on it).
val_loader = globals().get('VERIFY_LOADER')
if val_loader is None:
    val_loader = result['val_loader']

cpu = torch.device('cpu')
sub_model = sub.load_trained_model(cpu)
y_true = torch.cat([y.long() for _, y in val_loader])
y_pred = sub.predict(sub_model, val_loader, cpu)
rep = sub.classification_report(y_true, y_pred)

print('Submission file :', submission_path, f'({os.path.getsize(submission_path) / 1e6:.2f} MB)')
print('Conv2d layers   :', sum(isinstance(m, nn.Conv2d) for m in sub_model.modules()))
print(f"Validation      : acc={rep['accuracy']:.4f} macro_recall={rep['macro_recall']:.4f}")
print('CMS 35/35 target (>= 0.80) met on validation:', rep['macro_recall'] >= 0.80)

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except ImportError:
    print('Not on Colab - the submission file is at:', os.path.abspath(submission_path))